# Berkeley Haas Exec Ed
# Artficial Intellligence: Business Strategies and Applications
## Module 3 - Neural Networks and Deep Learning
## Building and deploying an image classification app using a pretrained Google deep learning model and deploying it on Hugging Face Spaces
James Gray, Learning Facilitator (jamesgray@berkeley.edu)

**Instructor**: James Gray  
**Web**: http://www.jamesgray.ai  
**Email**: james@jamesgray.ai  
**Discord**: https://discord.jamesgray.ai

## 1 - Install the Python packages for this demo

We will need the `transformers` package from Hugging Face to download and use machine learning models. The `transformers` API enable you to load, train, fine-tune and use transformers models.   

We will install `gradio` to build a simple UX for upload images to be classfied by the model

In [1]:
!pip install transformers # Hugging Face
!pip install gradio # simple UX for app
!pip install torch torchvision # PyTorch for deep learning


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


---

## 2 - Import Python libraries from the packages installed above
* We will be using a Google Vision Transformer model pre-trained with 14 million images with 21,843 classes at 224x224 resolution. You can learn more about it at the Hugging Face model card https://huggingface.co/google/vit-base-patch16-224
* We will also be using PIL (Python Imaging Library) for opening, saving, and manipulating images.

## ViTImageProcessor and ViTForImageClassification

- `ViTImageProcessor` is a class designed to preprocess images for input into **Vision Transformer (ViT)** models.
- It handles tasks like resizing, cropping, and normalizing images, ensuring the input is compatible with the requirements of the ViT model.

- `ViTForImageClassification` is a pre-trained Vision Transformer model class specifically fine-tuned for image classification tasks.

- This line of code sets up your environment to easily perform image classification using ViT models from the Hugging Face transformers library.

In [2]:
from transformers import ViTImageProcessor, ViTForImageClassification
## from PIL import Image #Python Imaging Library for maninpulating images
## import requests

/Users/jamesgray/My Drive (james@jamesgray.ai) (1)/Code/Berkeley-AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Python Imaging Library (PIL)

- This library provides tool for opening, manipulating, and saving images in various formats (JPEG, PNG, BMP, etc.).
- Imports the requests library, which is used for making HTTP requests in Python. This allows you to fetch images (or other resources) from the web via their URLs.

In [3]:
from PIL import Image #Python Imaging Library for maninpulating images
import requests

## 3 - Download a sample image on the web

- We will use this sample image to test our image classifier app.
- The image is an egyptian cat.

In [4]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

## 4 - Configure an image classifier processor
The processor will properly format the image such as resizing and normalizing it for what the model expects.

### `processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')`

This line of code is part of a workflow to prepare images for classification using a Vision Transformer (ViT) model. Let’s break it down step by step:

---

#### **1. What is `ViTImageProcessor`?**
- The `ViTImageProcessor` class comes from the **Hugging Face `transformers` library**.
- Its purpose is to **preprocess images** so they can be fed into a Vision Transformer (ViT) model.
- Preprocessing includes steps like:
  - **Resizing** the image to match the model's input size.
  - **Normalizing** pixel values (scaling them to a range suitable for the model).
  - Converting the image into the format required by the model (e.g., tensor or array).

---

#### **2. What does `from_pretrained` do?**
- The `from_pretrained` method downloads a **pre-trained image processor configuration** for the specified model.
- It ensures that the preprocessing is **compatible with the model architecture** and settings.
- In this case, the processor is being loaded for the model:
  - **`google/vit-base-patch16-224`**
  - This refers to a pre-trained Vision Transformer model with:
    - A base architecture.
    - Patch size of 16x16.
    - Input resolution of 224x224 pixels.

---

#### **3. Why do we need this line of code?**
- Models like ViT require input images to be in a very specific format.
- Without preprocessing, the model won’t work correctly or may produce errors.
- This line ensures:
  1. The image is resized to 224x224 pixels (the size required by the model).
  2. Pixel values are normalized and scaled appropriately.
  3. The image is ready for inference or training.

---

In [5]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

## 5 - Download the pretrained classifier model from Hugging Face

### `model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')`

This line of code loads a **pre-trained Vision Transformer (ViT) model** that is specifically fine-tuned for image classification tasks. Let’s break it down in simple terms:

---

#### **1. What is `ViTForImageClassification`?**
- `ViTForImageClassification` is a class from the Hugging Face `transformers` library.
- It represents a Vision Transformer (ViT) model that is designed for **image classification tasks**.
- The model processes an input image and predicts which category (or label) it belongs to, based on its training.

---

#### **2. What does `from_pretrained` do?**
- The `from_pretrained` method downloads a **pre-trained ViT model** from Hugging Face’s Model Hub.
- Pre-trained means the model has already been trained on a large dataset (e.g., ImageNet) to recognize general image patterns.
- In this case, the model being loaded is:
  - **`google/vit-base-patch16-224`**:
    - "base" refers to the model size (number of parameters).
    - "patch16" means the image is divided into 16x16 patches before being processed.
    - "224" refers to the input resolution of 224x224 pixels.

---

#### **3. Why use this line of code?**
- By loading a pre-trained model, you can skip the time-consuming and resource-intensive process of training a model from scratch.
- The pre-trained model can either:
  - Be used directly for inference (predicting labels for new images).
  - Be fine-tuned on a smaller dataset for a specific task.


In [6]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

---

## 6 - Process the image using the model processor and return PyTorch tensors


- process the image and return it as a tensor for the machine learning model
- the tensor is a multi-dimensional array that encapsulates the image data along with its shape (dimensions) and datatype.
- the tensor is what is used as the input to the machine learning model
- return_tensors="pt" indicates that the processor should return the processed data in the form of PyTorch tensors (hence "pt").
---

### `inputs = processor(images=image, return_tensors="pt")`

This line of code preprocesses an image and prepares it for input into a Vision Transformer (ViT) model. It uses the **image processor** to handle all the necessary transformations. Let’s break it down step by step:

---

#### **1. What Does the Code Do?**
- **`processor`**: Refers to a preloaded `ViTImageProcessor` object, which is responsible for preparing the image for the model.
- **`images=image`**: Passes the input image (e.g., loaded as a PIL image or NumPy array) to the processor for transformation.
- **`return_tensors="pt"`**: Converts the processed image into a PyTorch tensor format, which is required for models using the PyTorch backend.
- `inputs` is a Python dictionary object containing the processed data formatted as required by the model, including tensors and metadata.

---

#### **2. Why Is Preprocessing Necessary?**
Transformers, including Vision Transformers (ViT), require input data in a specific format. Raw images cannot be directly fed into the model. The preprocessing performed by `processor` includes:

1. **Resizing**:
   - The input image is resized to the dimensions required by the model (e.g., 224x224 pixels for `google/vit-base-patch16-224`).
   
2. **Normalization**:
   - Pixel values are normalized to match the scale expected by the model. Typically, this involves:
     - Scaling pixel values to a range (e.g., [0, 1]).
     - Applying mean and standard deviation normalization to align with the training data distribution.

3. **Tensor Conversion**:
   - The image is converted into a tensor format, which is the input format required by deep learning frameworks like PyTorch.

---

#### **3. Explanation of Parameters**
- **`images=image`**:
  - Specifies the image to be processed. This can be a single image or a batch of images.
  
- **`return_tensors="pt"`**:
  - Indicates that the output should be returned as a PyTorch tensor (`"pt"` stands for PyTorch).
  - Alternatives include `"tf"` for TensorFlow or `"np"` for NumPy arrays.

---



In [7]:
inputs = processor(images=image, return_tensors="pt")

## 7 - Inspect the tensor object to understand its structure
- `inputs` is a Python dictionary object containing the processed data formatted as required by the model, including tensors and metadata.
- 'pixel_values' is the key to a tensor object in the dictionary representing the preprocessed image, ready for input into the model
- The values in the tensor returned by the processor (e.g., inputs['pixel_values']) represent the pixel intensity values of the preprocessed image. These values are normalized and formatted in a way that the Vision Transformer (ViT) model expects.
- let's print it to see what it looks like
- the first dimension of the tensor (size 1) is the batch size, indicating there is one image in the batch
- the second dimension (size 3) indicates there are three color channels (e.g. RGB)
- the third dimension is the heigth of the image (224 pixels)
- the fourth dimension is the width of the image (224 pixels)


In [8]:
print("Tensors:", inputs)
# lets see what the shape is of the pixel_values tensor
tensor = inputs['pixel_values']
print ("The tensor shape for the dimensions are;", tensor.shape)

Tensors: {'pixel_values': tensor([[[[ 0.1137,  0.1686,  0.1843,  ..., -0.1922, -0.1843, -0.1843],
          [ 0.1373,  0.1686,  0.1843,  ..., -0.1922, -0.1922, -0.2078],
          [ 0.1137,  0.1529,  0.1608,  ..., -0.2314, -0.2235, -0.2157],
          ...,
          [ 0.8353,  0.7882,  0.7333,  ...,  0.7020,  0.6471,  0.6157],
          [ 0.8275,  0.7961,  0.7725,  ...,  0.5843,  0.4667,  0.3961],
          [ 0.8196,  0.7569,  0.7569,  ...,  0.0745, -0.0510, -0.1922]],

         [[-0.8039, -0.8118, -0.8118,  ..., -0.8902, -0.8902, -0.8980],
          [-0.7882, -0.7882, -0.7882,  ..., -0.8745, -0.8745, -0.8824],
          [-0.8118, -0.8039, -0.7882,  ..., -0.8902, -0.8902, -0.8902],
          ...,
          [-0.2706, -0.3176, -0.3647,  ..., -0.4275, -0.4588, -0.4824],
          [-0.2706, -0.2941, -0.3412,  ..., -0.4824, -0.5451, -0.5765],
          [-0.2784, -0.3412, -0.3490,  ..., -0.7333, -0.7804, -0.8353]],

         [[-0.5451, -0.4667, -0.4824,  ..., -0.7412, -0.6941, -0.7176],
    

## 8 - Run the model for it to predict the image class
* the model predicts one of the 1000 ImageNet classes.
* the "**inputs" unpacks the dictionary into keyword arguments for the model to use as its input for the model classification.
* "outputs" is a data structure containing the metadata and the logits of the prediction.
---

## Understanding Logits in Image Classification

When using a Vision Transformer (ViT) model for image classification, **logits** represent the model's raw predictions before any normalization. Here’s a simple breakdown:

---

#### **What Are Logits?**
- Logits are **raw, unprocessed scores** produced by the final layer of the model.
- Each value in the logits array corresponds to a specific class, indicating the model's confidence in predicting that class.
- **Important**: Logits are not probabilities; they need further processing (e.g., softmax) to interpret them as probabilities.

---

#### **Logits Structure**
- Logits are typically stored in a **1D array (or tensor)**.
- **For a single image**:
  - The shape of the logits tensor is `(1, number_of_classes)`:
    - **1 row**: Represents the image being classified.
    - **Columns**: Each column corresponds to one class.

Example:
- If the model is trained on ImageNet (1,000 classes), the logits array will have 1,000 values, one for each class.

---

#### **Interpreting Logits**
- The class with the **highest logit value** is the model's predicted class.


In [9]:
outputs = model(**inputs)
print("Model outputs:", outputs)

Model outputs: ImageClassifierOutput(loss=None, logits=tensor([[-2.7440e-01,  8.2152e-01, -8.3646e-02,  4.1588e-01,  5.6233e-01,
          1.8593e-01, -5.7729e-01, -4.6004e-01, -5.3389e-01,  2.4016e-01,
         -3.1957e-01, -5.9910e-01, -6.6403e-01, -4.9756e-01, -6.2448e-01,
         -1.3501e+00, -1.0016e-01, -6.2171e-01,  1.1087e-01, -1.1060e+00,
         -2.0846e-01,  3.1696e-01, -9.3153e-01, -3.0693e-01, -1.0124e+00,
         -1.8751e-01,  5.8825e-01, -3.6161e-01, -7.4697e-01,  7.4134e-01,
         -3.6653e-01, -2.7586e-01,  3.6596e-01, -1.1206e+00, -8.8844e-02,
         -1.1328e+00,  1.5458e-01, -1.0399e+00,  1.0136e+00, -1.0395e+00,
         -2.4214e+00,  5.1124e-01,  4.9458e-01, -7.4005e-01, -1.5815e+00,
         -3.2451e-01, -2.0448e+00, -4.8128e-01, -6.3616e-01, -1.1355e+00,
         -1.0902e+00, -4.5298e-02, -6.4045e-01, -2.3987e-01,  1.3110e-01,
         -1.2665e+00, -4.7161e-01, -4.3717e-01, -9.5664e-01, -5.9686e-01,
          5.0885e-01, -8.4840e-02,  2.6986e-01, -1.5041e-

## 9 - Return the logits of the model - the prediction for all classes
- get the logits from the model output - these are the raw output values produced by the last layer of the neural network model before any normalization or activation function (like softmax) is applied.
- these logits are the unnormalized predictions that are generated from the input data processed through the model network
---


In [10]:
logits = outputs.logits
print("Logits:", logits)
# the logits data structure is a 1D array of 1 row with 1000 columns that store the prediction for the class
print (logits.shape)

Logits: tensor([[-2.7440e-01,  8.2152e-01, -8.3646e-02,  4.1588e-01,  5.6233e-01,
          1.8593e-01, -5.7729e-01, -4.6004e-01, -5.3389e-01,  2.4016e-01,
         -3.1957e-01, -5.9910e-01, -6.6403e-01, -4.9756e-01, -6.2448e-01,
         -1.3501e+00, -1.0016e-01, -6.2171e-01,  1.1087e-01, -1.1060e+00,
         -2.0846e-01,  3.1696e-01, -9.3153e-01, -3.0693e-01, -1.0124e+00,
         -1.8751e-01,  5.8825e-01, -3.6161e-01, -7.4697e-01,  7.4134e-01,
         -3.6653e-01, -2.7586e-01,  3.6596e-01, -1.1206e+00, -8.8844e-02,
         -1.1328e+00,  1.5458e-01, -1.0399e+00,  1.0136e+00, -1.0395e+00,
         -2.4214e+00,  5.1124e-01,  4.9458e-01, -7.4005e-01, -1.5815e+00,
         -3.2451e-01, -2.0448e+00, -4.8128e-01, -6.3616e-01, -1.1355e+00,
         -1.0902e+00, -4.5298e-02, -6.4045e-01, -2.3987e-01,  1.3110e-01,
         -1.2665e+00, -4.7161e-01, -4.3717e-01, -9.5664e-01, -5.9686e-01,
          5.0885e-01, -8.4840e-02,  2.6986e-01, -1.5041e-03, -5.3330e-01,
          1.8371e-02,  1.3334e

## 10 - Find the largest index value that represents the likeliest class
* find the index of the highest value in the logits tensor, which corresponds to the most likely class according to the model.
* the "-1" of the argmax function specifies the last axis of the array. A 1D array has only one axis (axis 0), and in this case
* it will find the maximum value in the one row of 1,000 columns.

---
### `predicted_class_idx = logits.argmax(-1).item()`

This line of code extracts the **index of the predicted class** from the logits (raw prediction scores) generated by the model. Let’s break it down step by step:

---

#### **What Does This Code Do?**
1. **`logits.argmax(-1)`**:
   - Finds the **index of the maximum value** in the logits tensor along the last dimension (columns).
   - Each value in the logits tensor corresponds to a prediction score for a specific class. The highest score indicates the most likely class.

2. **`.item()`**:
   - Converts the result from a PyTorch tensor into a plain Python integer.

3. **Result**:
   - `predicted_class_idx` stores the index of the class with the highest prediction score. This index can be used to identify the predicted class.

---

#### **Why Is This Important?**
- The **logits** tensor is a raw output from the model. It contains prediction scores for all possible classes (e.g., 1,000 classes for ImageNet).
- To determine the model’s prediction, we need to find the **index** of the class with the highest score. This index represents the predicted class.

---

#### **Example**
Suppose the logits tensor contains the following values:
```python
logits = tensor([[1.2, 3.5, 0.7, 4.8, 2.1]])

logits.argmax(-1) finds the maximum value (4.8) and returns its index (3).

.item() converts this index into an integer.

predicted_class_idx = 3

In [11]:
predicted_class_idx = logits.argmax(-1).item()
# this returns an "index" into the 1D array of the maximum value. Let's see what number if returns in the array
print("The index of the predicted class is column:", predicted_class_idx)

# translate the predicted class index into a human-readable class name using a dictionary id2label provided by the model’s configuration.
# print("Predicted class:", model.config.id2label[predicted_class_idx])

The index of the predicted class is column: 285


## 11 - Translate Predicated Class Index into a Readable Class Name

`model.config.id2label`
- This is a Python dictionary stored in the model’s configuration.
- It maps numerical class indices (e.g., 0, 1, 2, …) to meaningful, human-readable class labels (e.g., "cat", "dog", "airplane").

`model.config.id2label[predicted_class_idx]`
- uses the predicted class index (predicted_class_idx) as a key to look up its corresponding label in the dictionary.
- Retrieves the descriptive label associated with that index.

`print("Predicted class:", ...)`
- Displays the human-readable class name to the user.

In [12]:
# translate the predicted class index into a human-readable class name using a dictionary id2label provided by the model’s configuration.
# print("Predicted class:", model.config.id2label[4])
print("Predicted class:", model.config.id2label[predicted_class_idx])

Predicted class: Egyptian cat


## All up Code Above

In [ ]:
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image #Python Imaging Library for maninpulating images
import requests

# this is an example using an image on the web
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)


# build the image classifier processor and model - the processor will properly format the image such as resize and normalize.
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
# download the pre-trained image classifier model from the Hugging Face site
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

# process the image and return it as a tensor for the machine learning model.
# the tensor is a multi-dimensional array that encapsulates the image data along with its shape(dimensions) and datatype.
# the tensor is what is used as the input to the machine learning model
# return_tensors="pt" indicates that the processor should return the processed data in the form of PyTorch tensors (hence "pt").
inputs = processor(images=image, return_tensors="pt")

# inputs is a Python dictionary object containing the processed data formatted as required by the model, including tensors and metadata.
# let's print it to see what it looks like
print("Tensors:", inputs)
# lets see what the shape is of the pixel_values tensor
tensor = inputs['pixel_values']
print ("The tensor shape for the dimensions are;", tensor.shape)

# the first dimension of the tensor (size 1) is the batch size, indicating there is one image in the batch
# the second dimension (size 3) indicates there are three color channels (e.g. RGB)
# the third dimension is the heigth of the image (24 pixels)
# the fourth dimension is the width of the image (24 pixels)

# model predicts one of the 1000 ImageNet classes.
# the "**inputs" unpacks the dictionary into keyword arguments for the model to use as its input for the model classification.
# "outputs" is a data structure containing the metadata and the logits of the prediction.
outputs = model(**inputs)
print("Model outputs:", outputs)

# get the logits from the model output - these are the raw output values produced by the last layer of the neural
# network model before any normalization or activation function (like softmax) is applied.
# these logits are the unnormalized predictions that are generated from the input data processed through the model network
logits = outputs.logits
print("Logits:", logits)
# the logits data structure is a 1D array of 1 row with 1000 columns that store the prediction for the class
print (logits.shape)


# find the index of the highest value in the logits tensor, which corresponds to the most likely class according to the model.
# the "-1" of the argmax function specifies the last axis of the array. A 1D array has only one axis (axis 0), and in this case
# it will find the maximum value in the one row of 1,000 columns.
predicted_class_idx = logits.argmax(-1).item()
# this returns an "index" into the 1D array of the maximum value. Let's see what number if returns in the array
print("The index of the predicted class is column:", predicted_class_idx)

# translate the predicted class index into a human-readable class name using a dictionary id2label provided by the model’s configuration.
print("Predicted class:", model.config.id2label[predicted_class_idx])


## Build a simple UX using Gradio to upload images to the classifier

In [ ]:
import os
import gradio as gr

In [ ]:
# build a function to process the image and return a predicted class
def get_image_classification (pil_image):
  inputs = processor(images=pil_image, return_tensors="pt")
  outputs = model(**inputs)
  logits = outputs.logits
  predicted_class_idx = logits.argmax(-1).item()
  classLabel = model.config.id2label[predicted_class_idx]
  return classLabel

In [ ]:
# build the gradio demo UI
demo = gr.Interface(
    fn=get_image_classification,
    inputs=gr.Image(type="pil"),
    outputs=["text"]
)

In [ ]:
# let's launch the UX!
demo.launch()